<a href="https://colab.research.google.com/github/wxhfy/wxhfy/blob/main/batch/AlphaFold2_batch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ColabFold v1.5.5: AlphaFold2 w/ MMseqs2 BATCH

<img src="https://raw.githubusercontent.com/sokrypton/ColabFold/main/.github/ColabFold_Marv_Logo_Small.png" height="256" align="right" style="height:256px">

Easy to use AlphaFold2 protein structure [(Jumper et al. 2021)](https://www.nature.com/articles/s41586-021-03819-2) and complex [(Evans et al. 2021)](https://www.biorxiv.org/content/10.1101/2021.10.04.463034v1) prediction using multiple sequence alignments generated through MMseqs2. For details, refer to our manuscript:

[Mirdita M, Schütze K, Moriwaki Y, Heo L, Ovchinnikov S, Steinegger M. ColabFold: Making protein folding accessible to all.
*Nature Methods*, 2022](https://www.nature.com/articles/s41592-022-01488-1)

**Usage**

`input_dir` directory with only fasta files or MSAs stored in Google Drive. MSAs need to be A3M formatted and have an `.a3m` extention. For MSAs MMseqs2 will not be called.

`result_dir` results will be written to the result directory in Google Drive

Old versions: [v1.4](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.4.0/batch/AlphaFold2_batch.ipynb), [v1.5.1](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.1/batch/AlphaFold2_batch.ipynb), [v1.5.2](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.2/batch/AlphaFold2_batch.ipynb), [v1.5.3-patch](https://colab.research.google.com/github/sokrypton/ColabFold/blob/56c72044c7d51a311ca99b953a71e552fdc042e1/batch/AlphaFold2_batch.ipynb)

<strong>For more details, see <a href="#Instructions">bottom</a> of the notebook and checkout the [ColabFold GitHub](https://github.com/sokrypton/ColabFold). </strong>

-----------

### News
- <b><font color='green'>2023/07/31: The ColabFold MSA server is back to normal. It was using older DB (UniRef30 2202/PDB70 220313) from 27th ~8:30 AM CEST to 31st ~11:10 AM CEST.</font></b>
- <b><font color='green'>2023/06/12: New databases! UniRef30 updated to 2023_02 and PDB to 230517. We now use PDB100 instead of PDB70 (see notes in the [main](https://colabfold.com) notebook).</font></b>
- <b><font color='green'>2023/06/12: We introduced a new default pairing strategy: Previously, for multimer predictions with more than 2 chains, we only pair if all sequences taxonomically match ("complete" pairing). The new default "greedy" strategy pairs any taxonomically matching subsets.</font></b>

In [1]:
#@title Mount google drive
from google.colab import drive
drive.mount('/content/drive')
from sys import version_info
python_version = f"{version_info.major}.{version_info.minor}"

Mounted at /content/drive


In [1]:
#@title Input protein sequence, then hit `Runtime` -> `Run all`

input_dirs = ['/content/drive/MyDrive/benchmark1'] #@param {type:"raw"}
result_dir = '/content/drive/MyDrive/result' #@param {type:"string"}

# number of models to use
#@markdown ---
#@markdown ### Advanced settings
msa_mode = "MMseqs2 (UniRef+Environmental)" #@param ["MMseqs2 (UniRef+Environmental)", "MMseqs2 (UniRef only)","single_sequence","custom"]
num_models = 5 #@param [1,2,3,4,5] {type:"raw"}
num_recycles = 3 #@param [1,3,6,12,24,48] {type:"raw"}
stop_at_score = 100 #@param {type:"string"}
#@markdown - early stop computing models once score > threshold (avg. plddt for "structures" and ptmscore for "complexes")
use_custom_msa = False
num_relax = 0 #@param [0, 1, 5] {type:"raw"}
use_amber = num_relax > 0
relax_max_iterations = 200 #@param [0,200,2000] {type:"raw"}
use_templates = False #@param {type:"boolean"}
do_not_overwrite_results = True #@param {type:"boolean"}
zip_results = False #@param {type:"boolean"}


In [3]:
#@title Install dependencies
%%bash -s $use_amber $use_templates $python_version

set -e

USE_AMBER=$1
USE_TEMPLATES=$2
PYTHON_VERSION=$3

if [ ! -f COLABFOLD_READY ]; then
  # install dependencies
  # We have to use "--no-warn-conflicts" because colab already has a lot preinstalled with requirements different to ours
  pip install -q --no-warn-conflicts "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold"
  if [ -n "${TPU_NAME}" ]; then
    pip install -q --no-warn-conflicts -U dm-haiku==0.0.10 jax==0.3.25
  fi
  ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold
  ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold
  # hack to fix TF crash
  rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so
  touch COLABFOLD_READY
fi

# Download params (~1min)
python -m colabfold.download

# setup conda
if [ ${USE_AMBER} == "True" ] || [ ${USE_TEMPLATES} == "True" ]; then
  if [ ! -f CONDA_READY ]; then
    wget -qnc https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-25.3.1-0-Linux-x86_64.sh
    bash Miniforge3-25.3.1-0-Linux-x86_64.sh -bfp /usr/local 2>&1 1>/dev/null
    rm Miniforge3-25.3.1-0-Linux-x86_64.sh
    conda config --set auto_update_conda false
    touch CONDA_READY
  fi
fi
# setup template search
if [ ${USE_TEMPLATES} == "True" ] && [ ! -f HH_READY ]; then
  conda install -y -q -c conda-forge -c bioconda kalign2=2.04 hhsuite=3.3.0 python="${PYTHON_VERSION}" 2>&1 1>/dev/null
  touch HH_READY
fi
# setup openmm for amber refinement
if [ ${USE_AMBER} == "True" ] && [ ! -f AMBER_READY ]; then
  conda install -y -q -c conda-forge openmm=8.2.0 python="${PYTHON_VERSION}" pdbfixer 2>&1 1>/dev/null
  touch AMBER_READY
fi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.4/248.4 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.8/373.8 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.0/259.0 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 6.9 MB/s eta 0:00:00


  file.extractall(path=params_dir)


In [ ]:
#@title Run Prediction
import sys
from colabfold.batch import get_queries, run
from colabfold.download import default_data_dir
from colabfold.utils import setup_logging
from pathlib import Path
import glob

# For some reason we need that to get pdbfixer to import
if use_amber and f"/usr/local/lib/python{python_version}/site-packages/" not in sys.path:
    sys.path.insert(0, f"/usr/local/lib/python{python_version}/site-packages/")

for input_dir in input_dirs:
    input_path = Path(input_dir)
    # fasta_files = glob.glob(str(input_path / "*.fasta"))
    fasta_files = ["/content/drive/MyDrive/benchmark1/DECOY_eval.fasta", "/content/drive/MyDrive/benchmark1/DECOY_test.fasta", "/content/drive/MyDrive/benchmark1/DECOY_train.fasta"]
    for fasta_file in fasta_files:
        queries, is_complex = get_queries(fasta_file) # Process each fasta file individually
        # Create a specific result directory for each fasta file
        fasta_name = Path(fasta_file).stem
        current_result_dir = Path(result_dir).joinpath(input_path.name, fasta_name)

        setup_logging(current_result_dir.joinpath("log.txt"))

        run(
            queries=queries,
            result_dir=current_result_dir,
            use_templates=use_templates,
            num_relax=num_relax,
            relax_max_iterations=relax_max_iterations,
            msa_mode=msa_mode,
            model_type="auto",
            num_models=num_models,
            num_recycles=num_recycles,
            model_order=[1, 2, 3, 4, 5],
            is_complex=is_complex,
            data_dir=default_data_dir,
            keep_existing_results=do_not_overwrite_results,
            rank_by="auto",
            pair_mode="unpaired+paired",
            pairing_strategy="greedy", # changed from original notebook
            stop_at_score=stop_at_score,
            zip_results=zip_results,
            user_agent="colabfold/google-colab-batch",
        )


2025-11-12 06:32:02,978 Running on GPU
2025-11-12 06:32:04,225 Found 5 citations for tools or databases
2025-11-12 06:32:04,227 Skipping UniRef50_Q9NZN9 (already done)
2025-11-12 06:32:04,228 Skipping UniRef50_P51956 (already done)
2025-11-12 06:32:04,229 Skipping UniRef50_P51480 (already done)
2025-11-12 06:32:04,229 Skipping UniRef50_O74424 (already done)
2025-11-12 06:32:04,230 Skipping UniRef50_A7ZIA5 (already done)
2025-11-12 06:32:04,231 Skipping UniRef50_B0TA39 (already done)
2025-11-12 06:32:04,231 Skipping UniRef50_A8FLU4 (already done)
2025-11-12 06:32:04,232 Skipping UniRef50_Q5AD49 (already done)
2025-11-12 06:32:04,233 Skipping UniRef50_Q63ZY7 (already done)
2025-11-12 06:32:04,234 Skipping UniRef50_O04350 (already done)
2025-11-12 06:32:04,235 Skipping UniRef50_Q99210 (already done)
2025-11-12 06:32:04,235 Skipping UniRef50_P59425 (already done)
2025-11-12 06:32:04,236 Skipping UniRef50_P42842 (already done)
2025-11-12 06:32:04,237 Skipping UniRef50_Q8EW68 (already done)


PENDING:   0%|          | 0/150 [elapsed: 00:00 remaining: ?]

2025-11-12 06:34:13,401 Sleeping for 10s. Reason: PENDING


COMPLETE: 100%|██████████| 150/150 [elapsed: 00:10 remaining: 00:00]


2025-11-12 06:34:25,669 Padding length to 33
2025-11-12 06:34:29,354 alphafold2_ptm_model_1_seed_000 recycle=0 pLDDT=73.2 pTM=0.2
2025-11-12 06:34:32,911 alphafold2_ptm_model_1_seed_000 recycle=1 pLDDT=74 pTM=0.202 tol=0.321
2025-11-12 06:34:36,451 alphafold2_ptm_model_1_seed_000 recycle=2 pLDDT=73.1 pTM=0.197 tol=0.267
2025-11-12 06:34:40,030 alphafold2_ptm_model_1_seed_000 recycle=3 pLDDT=73 pTM=0.196 tol=0.173
2025-11-12 06:34:40,031 alphafold2_ptm_model_1_seed_000 took 14.4s (3 recycles)
2025-11-12 06:34:43,624 alphafold2_ptm_model_2_seed_000 recycle=0 pLDDT=73.2 pTM=0.208
2025-11-12 06:34:47,207 alphafold2_ptm_model_2_seed_000 recycle=1 pLDDT=74.4 pTM=0.216 tol=0.316
2025-11-12 06:34:50,788 alphafold2_ptm_model_2_seed_000 recycle=2 pLDDT=74.4 pTM=0.214 tol=0.177
2025-11-12 06:34:54,371 alphafold2_ptm_model_2_seed_000 recycle=3 pLDDT=74.4 pTM=0.215 tol=0.168
2025-11-12 06:34:54,372 alphafold2_ptm_model_2_seed_000 took 14.3s (3 recycles)
2025-11-12 06:34:57,969 alphafold2_ptm_model_

...

2025-11-12 06:38:43,584 Padding length to 33
2025-11-12 06:38:47,314 alphafold2_ptm_model_1_seed_000 recycle=0 pLDDT=74.5 pTM=0.234
2025-11-12 06:38:50,895 alphafold2_ptm_model_1_seed_000 recycle=1 pLDDT=73.6 pTM=0.24 tol=0.299
2025-11-12 06:38:54,478 alphafold2_ptm_model_1_seed_000 recycle=2 pLDDT=73.3 pTM=0.239 tol=0.246
2025-11-12 06:38:58,069 alphafold2_ptm_model_1_seed_000 recycle=3 pLDDT=73.4 pTM=0.24 tol=0.0878
2025-11-12 06:38:58,070 alphafold2_ptm_model_1_seed_000 took 14.5s (3 recycles)
2025-11-12 06:39:01,673 alphafold2_ptm_model_2_seed_000 recycle=0 pLDDT=73.2 pTM=0.25
2025-11-12 06:39:05,256 alphafold2_ptm_model_2_seed_000 recycle=1 pLDDT=72.1 pTM=0.249 tol=0.411
2025-11-12 06:39:08,841 alphafold2_ptm_model_2_seed_000 recycle=2 pLDDT=71.2 pTM=0.248 tol=0.298
2025-11-12 06:39:12,419 alphafold2_ptm_model_2_seed_000 recycle=3 pLDDT=71.6 pTM=0.25 tol=0.2
2025-11-12 06:39:12,420 alphafold2_ptm_model_2_seed_000 took 14.3s (3 recycles)
2025-11-12 06:39:16,020 alphafold2_ptm_model

PENDING:   0%|          | 0/150 [elapsed: 00:00 remaining: ?]

2025-11-12 06:39:56,805 Sleeping for 5s. Reason: PENDING


COMPLETE: 100%|██████████| 150/150 [elapsed: 00:05 remaining: 00:00]


2025-11-12 06:40:04,058 Padding length to 33
2025-11-12 06:40:07,680 alphafold2_ptm_model_1_seed_000 recycle=0 pLDDT=61.7 pTM=0.136
2025-11-12 06:40:11,178 alphafold2_ptm_model_1_seed_000 recycle=1 pLDDT=62.4 pTM=0.131 tol=0.655
2025-11-12 06:40:14,682 alphafold2_ptm_model_1_seed_000 recycle=2 pLDDT=61.7 pTM=0.129 tol=0.875
2025-11-12 06:40:18,197 alphafold2_ptm_model_1_seed_000 recycle=3 pLDDT=61.9 pTM=0.128 tol=0.404
2025-11-12 06:40:18,198 alphafold2_ptm_model_1_seed_000 took 14.1s (3 recycles)
2025-11-12 06:40:21,760 alphafold2_ptm_model_2_seed_000 recycle=0 pLDDT=60.8 pTM=0.13
2025-11-12 06:40:25,322 alphafold2_ptm_model_2_seed_000 recycle=1 pLDDT=63.2 pTM=0.124 tol=1.59
2025-11-12 06:40:28,900 alphafold2_ptm_model_2_seed_000 recycle=2 pLDDT=64.1 pTM=0.142 tol=0.244
2025-11-12 06:40:32,485 alphafold2_ptm_model_2_seed_000 recycle=3 pLDDT=66 pTM=0.16 tol=0.301
2025-11-12 06:40:32,485 alphafold2_ptm_model_2_seed_000 took 14.3s (3 recycles)
2025-11-12 06:40:36,085 alphafold2_ptm_model

In [ ]:
import tensorflow as tf

gpu_available = tf.config.list_physical_devices('GPU')
if gpu_available:
    print("GPU is available.")
else:
    print("GPU is NOT available. Please change your runtime type to GPU.")

# Instructions <a name="Instructions"></a>
**Quick start**
1. Upload your single fasta files to a folder in your Google Drive
2. Define path to the fold containing the fasta files (`input_dir`) define an outdir (`output_dir`)
3. Press "Runtime" -> "Run all".

**Result zip file contents**

At the end of the job a all results `jobname.result.zip` will be uploaded to your (`output_dir`) Google Drive. Each zip contains one protein.

1. PDB formatted structures sorted by avg. pIDDT. (unrelaxed and relaxed if `use_amber` is enabled).
2. Plots of the model quality.
3. Plots of the MSA coverage.
4. Parameter log file.
5. A3M formatted input MSA.
6. BibTeX file with citations for all used tools and databases.


**Troubleshooting**
* Check that the runtime type is set to GPU at "Runtime" -> "Change runtime type".
* Try to restart the session "Runtime" -> "Factory reset runtime".
* Check your input sequence.

**Known issues**
* Google Colab assigns different types of GPUs with varying amount of memory. Some might not have enough memory to predict the structure for a long sequence.
* Google Colab assigns different types of GPUs with varying amount of memory. Some might not have enough memory to predict the structure for a long sequence.
* Your browser can block the pop-up for downloading the result file. You can choose the `save_to_google_drive` option to upload to Google Drive instead or manually download the result file: Click on the little folder icon to the left, navigate to file: `jobname.result.zip`, right-click and select \"Download\" (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).

**Limitations**
* Computing resources: Our MMseqs2 API can handle ~20-50k requests per day.
* MSAs: MMseqs2 is very precise and sensitive but might find less hits compared to HHblits/HMMer searched against BFD or Mgnify.
* We recommend to additionally use the full [AlphaFold2 pipeline](https://github.com/deepmind/alphafold).

**Description of the plots**
*   **Number of sequences per position** - We want to see at least 30 sequences per position, for best performance, ideally 100 sequences.
*   **Predicted lDDT per position** - model confidence (out of 100) at each position. The higher the better.
*   **Predicted Alignment Error** - For homooligomers, this could be a useful metric to assess how confident the model is about the interface. The lower the better.

**Bugs**
- If you encounter any bugs, please report the issue to https://github.com/sokrypton/ColabFold/issues

**License**

The source code of ColabFold is licensed under [MIT](https://raw.githubusercontent.com/sokrypton/ColabFold/main/LICENSE). Additionally, this notebook uses AlphaFold2 source code and its parameters licensed under [Apache 2.0](https://raw.githubusercontent.com/deepmind/alphafold/main/LICENSE) and  [CC BY 4.0](https://creativecommons.org/licenses/by-sa/4.0/) respectively. Read more about the AlphaFold license [here](https://github.com/deepmind/alphafold).

**Acknowledgments**
- We thank the AlphaFold team for developing an excellent model and open sourcing the software.

- Do-Yoon Kim for creating the ColabFold logo.

- A colab by Sergey Ovchinnikov ([@sokrypton](https://twitter.com/sokrypton)), Milot Mirdita ([@milot_mirdita](https://twitter.com/milot_mirdita)) and Martin Steinegger ([@thesteinegger](https://twitter.com/thesteinegger)).
